# NFL Weekly Game Predictor

A machine learning pipeline that predicts the winner of each NFL game, with a win probability for every matchup. 

Our approach:
1. Baseline: An ELO rating model, benchmarked against "always pick home" and the Vegas line
2. Features: Rolling team performances (EPA per play on offense and defense)
3. Models: Logistic Regression to predict probability outcome easily, then Gradient Boosting for non-linear effects and interactions between features.
4. Evaluation:  Walk-forward testing by season, so the  model never sees future games
- Metrics: accuracy,  log loss, and brier score
- Data: nflverse ('nflreadpy'), 2015-2026
- Future: weekly pipeline and ATS predictions

## 1. Setup & Loading Data

In [2]:
import sys
# Importing our own file from the src folder 
sys.path.append("src")

import nflreadpy as nfl
from elo_baseline import run_elo, evaluate, TEAM_FIX

In [3]:
# One row per game: teams, scores, spread
games = nfl.load_schedules(seasons=range(2015, 2027)).to_pandas()

# One row per team per game: EPA, sacks
stats = nfl.load_team_stats(seasons=range(2015, 2027)).to_pandas()

## 2. ELO Baseline
- Each team gets a rating that's updated after every game.
- Predictions are made *before* the result updates the ratings, so we don't have leakage. 
- Scored against "always home" and Vegas 

In [4]:
# Run Elo through every game in date order; adds elo_home, elo_away, elo_prob columns
df, final_ratings = run_elo(games)

# Compare Elo vs "always pick home" vs Vegas (regular-season games from 2018 on)
evaluate(df, start_season=2018)

,games,accuracy,log_loss,brier
Always home,2197.0,0.539827,0.691329,0.249084
Elo,2197.0,0.633591,0.637004,0.222963
Vegas (spread-implied),2197.0,0.663177,0.611373,0.211710


## 3. Feature engineering
- Rolling 6-game EPA per play for offense and defense.
- `shift(1)` keeps each game out of its own average, so the model only sees information available before kickoff.

In [5]:
s = stats.copy()

# EPA per play = total EPA / number of plays (pass attempts + rushes + sacks taken)
plays = s["attempts"] + s["carries"] + s["sacks_suffered"]
s["off_epa_pp"] = (s["passing_epa"] + s["rushing_epa"]) / plays

# Defense = what the opponent's offense did against this team in the same game
# (lower is better for the defense)
opp = s[["game_id", "team", "off_epa_pp"]].rename(
    columns={"team": "opponent_team", "off_epa_pp": "def_epa_pp"})
s = s.merge(opp, on=["game_id", "opponent_team"], how="left")

In [6]:
# Sort each team's games in time order so "previous games" means what we want
s = s.sort_values(["team", "season", "week"])

# Rolling average of the previous 6 games for each team.
# shift(1) drops the current game from its own average (prevents leakage).
# min_periods=1 lets early-season games use whatever games exist so far.
for col in ["off_epa_pp", "def_epa_pp"]:
    s[col + "_r6"] = (s.groupby("team")[col]
                       .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean()))

In [7]:
# Use the same team abbreviations as the Elo table (e.g. OAK -> LV)
s["team"] = s["team"].replace(TEAM_FIX)

# Pull each team's rolling numbers out twice: once as the home team, once as the away team
home = s[["game_id", "team", "off_epa_pp_r6", "def_epa_pp_r6"]].rename(
    columns={"team": "home_team", "off_epa_pp_r6": "home_off", "def_epa_pp_r6": "home_def"})
away = s[["game_id", "team", "off_epa_pp_r6", "def_epa_pp_r6"]].rename(
    columns={"team": "away_team", "off_epa_pp_r6": "away_off", "def_epa_pp_r6": "away_def"})

# Final table: one row per game with Elo plus both teams' rolling features.
# Missing values are expected for upcoming games and each team's first game.
data = (df.merge(home, on=["game_id", "home_team"], how="left")
          .merge(away, on=["game_id", "away_team"], how="left"))

Pipeline Check

In [8]:
gid = s["game_id"].iloc[0]
s[s["game_id"] == gid][["team", "opponent_team", "off_epa_pp", "def_epa_pp"]]

,team,opponent_team,off_epa_pp,def_epa_pp
0,ARI,NO,0.132297,-0.027431
22,NO,ARI,-0.027431,0.132297


In [9]:
g = s[s["team"] == "KC"].head(2)
print(g[["season", "week", "off_epa_pp", "off_epa_pp_r6"]])
print("OK" if abs(g["off_epa_pp_r6"].iloc[1] - g["off_epa_pp"].iloc[0]) < 1e-9 else "PROBLEM")

    season  week  off_epa_pp  off_epa_pp_r6
15    2015     1   -0.027574            NaN
47    2015     2   -0.191614      -0.027574
OK


In [10]:
print("Same number of games:", len(data) == len(df))
data[["home_off", "home_def", "away_off", "away_def"]].isna().sum()

Same number of games: True


home_off    209
home_def    209
away_off    209
away_def    209
dtype: int64

## 4. Model Training

### Logistic Regression
- Each input is the home team - away team, so a positive number equates to the home team advantage
- Walk Forward Testing: model trains only on earlier seasons and predict that season. 
- Logistic regression outputs a win probability directly. It's less likely to overfit than a complex model.

In [11]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from elo_baseline import _metrics, vegas_prob

In [12]:
# Difference between home and away for each input (positive = home team is stronger)
data["elo_diff"] = data["elo_home"] - data["elo_away"]
data["off_diff"] = data["home_off"] - data["away_off"]
data["def_diff"] = data["home_def"] - data["away_def"]
feats = ["elo_diff", "off_diff", "def_diff"]

In [13]:
# Keep only finished games with complete features, and drop ties
played = data.dropna(subset=feats + ["home_score", "away_score"])
played = played[played["home_score"] != played["away_score"]].copy()
played["home_win"] = (played["home_score"] > played["away_score"]).astype(int)

In [14]:
# Walk-forward: for each season, train only on earlier seasons, then predict that season
preds = []
for season in range(2018, 2026):
    train = played[played["season"] < season]
    test = played[(played["season"] == season) & (played["game_type"] == "REG")].copy()
    model = LogisticRegression().fit(train[feats], train["home_win"])
    test["lr_prob"] = model.predict_proba(test[feats])[:, 1]
    preds.append(test)


In [15]:
# Score Elo and the logistic model on the exact same games
res = pd.concat(preds)
y = res["home_win"].to_numpy()
pd.DataFrame({"Elo": _metrics(y, res["elo_prob"]),
              "Logistic": _metrics(y, res["lr_prob"])}).T

,games,accuracy,log_loss,brier
Elo,2119.0,0.633790,0.637043,0.222939
Logistic,2119.0,0.641812,0.634437,0.221937


### Adding rest-day feature

In [16]:
# Rest advantage in days (positive = home team had more rest before this game)
data["rest_diff"] = data["home_rest"] - data["away_rest"]

In [17]:
def walk_forward(feats):
    """Same walk-forward test as before, packaged so we can try different inputs."""
    # Also require a spread so every model is scored on the same games as Vegas
    p = data.dropna(subset=feats + ["home_score", "away_score", "spread_line"])
    p = p[p["home_score"] != p["away_score"]].copy()
    p["home_win"] = (p["home_score"] > p["away_score"]).astype(int)

    out = []
    for season in range(2018, 2026):
        train = p[p["season"] < season]  # only the past
        test = p[(p["season"] == season) & (p["game_type"] == "REG")].copy()
        model = LogisticRegression().fit(train[feats], train["home_win"])
        test["prob"] = model.predict_proba(test[feats])[:, 1]
        out.append(test)
    return pd.concat(out)

In [18]:
base = walk_forward(["elo_diff", "off_diff", "def_diff"])               # what we had
more = walk_forward(["elo_diff", "off_diff", "def_diff", "rest_diff"])  # plus rest days

y = more["home_win"].to_numpy()
pd.DataFrame({
    "Elo": _metrics(y, more["elo_prob"]),
    "Logistic (3 features)": _metrics(base["home_win"].to_numpy(), base["prob"]),
    "Logistic + rest": _metrics(y, more["prob"]),
    "Vegas": _metrics(y, vegas_prob(more["spread_line"])),
}).T

,games,accuracy,log_loss,brier
Elo,2119.0,0.633790,0.637043,0.222939
Logistic (3 features),2119.0,0.641812,0.634437,0.221937
Logistic + rest,2119.0,0.640868,0.634712,0.222168
Vegas,2119.0,0.663992,0.610020,0.211111


### **Takeaway** 
- Rest days added nothing since Elo and EPA captured what was already shown
- We want to reach the Vegas benchmark. The gap from our model to Vegas is most likely missing feature information

### Gradient-Boosting 

In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [20]:
# Same four inputs as the logistic model, so any difference comes from the model, not the data
feats = ["elo_diff", "off_diff", "def_diff", "rest_diff"]

# Keep finished games with complete features and a spread; drop ties (no winner)
p = data.dropna(subset=feats + ["home_score", "away_score", "spread_line"])
p = p[p["home_score"] != p["away_score"]].copy()
p["home_win"] = (p["home_score"] > p["away_score"]).astype(int)

In [21]:
out = []
for season in range(2018, 2026):
    # Walk-forward: train only on earlier seasons, test on this one
    train = p[p["season"] < season]
    test = p[(p["season"] == season) & (p["game_type"] == "REG")].copy()

    # Gradient boosting builds many small decision trees, each one correcting
    # the mistakes of the ones before it.
    # max_depth=2: very shallow trees, so each can only capture simple patterns
    # learning_rate=0.05: small steps, so no single tree has too much influence
    # max_iter=100: number of trees
    # These settings limit overfitting, since we only have ~270 games per season
    gb = HistGradientBoostingClassifier(max_depth=2, learning_rate=0.05,
                                        max_iter=100, random_state=0)
    gb.fit(train[feats], train["home_win"])

    # Predicted probability that the home team wins
    test["gb_prob"] = gb.predict_proba(test[feats])[:, 1]
    out.append(test)

In [22]:
# Combine all test seasons into one table
gbres = pd.concat(out)
y = gbres["home_win"].to_numpy()  # actual outcomes

# Score on the same games: previous best logistic model, gradient boosting, and Vegas
pd.DataFrame({
    "Logistic + rest": _metrics(y, more["prob"]),
    "Gradient boosting": _metrics(y, gbres["gb_prob"]),
    "Vegas": _metrics(y, vegas_prob(gbres["spread_line"])),
}).T

,games,accuracy,log_loss,brier
Logistic + rest,2119.0,0.640868,0.634712,0.222168
Gradient boosting,2119.0,0.636621,0.639137,0.224214
Vegas,2119.0,0.663992,0.610020,0.211111


### **Takeaway**
- Gradient Boosting is a more flexible model in comparison to Logistic Regression, however it didn't beat it. 
- Bottleneck is the quality of the inputs instead of the model. 

### Adding richer team features

In [23]:
# Split EPA per play into passing and rushing (offense)
s["pass_epa_pp"] = s["passing_epa"] / (s["attempts"] + s["sacks_suffered"])
s["rush_epa_pp"] = s["rushing_epa"] / s["carries"]

# Defense = what the opponent's offense did in the same game (lower is better)
s["opponent_team"] = s["opponent_team"].replace(TEAM_FIX)  # same abbreviations as "team"
opp = s[["game_id", "team", "pass_epa_pp", "rush_epa_pp"]].rename(
    columns={"team": "opponent_team",
             "pass_epa_pp": "def_pass_epa_pp", "rush_epa_pp": "def_rush_epa_pp"})
s = s.merge(opp, on=["game_id", "opponent_team"], how="left")

In [24]:
# Rolling average of the previous 6 games; shift(1) keeps each game out of its own average
new_cols = ["pass_epa_pp", "rush_epa_pp", "def_pass_epa_pp", "def_rush_epa_pp"]
s = s.sort_values(["team", "season", "week"])
for col in new_cols:
    s[col + "_r6"] = (s.groupby("team")[col]
                       .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean()))

In [25]:
# Attach to the game table, once for the home team and once for the away team
r6 = [c + "_r6" for c in new_cols]
home = s[["game_id", "team"] + r6].rename(
    columns={"team": "home_team", **{c: "home_" + c for c in r6}})
away = s[["game_id", "team"] + r6].rename(
    columns={"team": "away_team", **{c: "away_" + c for c in r6}})
data = (data.merge(home, on=["game_id", "home_team"], how="left")
            .merge(away, on=["game_id", "away_team"], how="left"))

# Home-minus-away gaps (positive = home team is stronger on that measure)
for c in new_cols:
    data[c + "_diff"] = data["home_" + c + "_r6"] - data["away_" + c + "_r6"]

In [26]:
# Elo plus the four split-EPA gaps, same walk-forward test as before
feats2 = ["elo_diff", "pass_epa_pp_diff", "rush_epa_pp_diff",
          "def_pass_epa_pp_diff", "def_rush_epa_pp_diff"]
split = walk_forward(feats2)

pd.DataFrame({
    "Logistic + rest": _metrics(more["home_win"].to_numpy(), more["prob"]),
    "Logistic + split EPA": _metrics(split["home_win"].to_numpy(), split["prob"]),
    "Vegas": _metrics(split["home_win"].to_numpy(), vegas_prob(split["spread_line"])),
}).T

,games,accuracy,log_loss,brier
Logistic + rest,2119.0,0.640868,0.634712,0.222168
Logistic + split EPA,2119.0,0.633790,0.635297,0.222388
Vegas,2119.0,0.663992,0.610020,0.211111


### **Takeaway**
- Log loss unchcanged meaning that the splitting was insignificant
- Since all these additional features aren't helping the model, it may be game-specific criterias

### Vegas Line Testing

In [27]:
# Model 1: the Vegas spread alone, recalibrated into a probability by logistic regression
vegas_only = walk_forward(["spread_line"])

# Model 2: the spread plus our own features (Elo, offense, defense gaps)
vegas_plus = walk_forward(["spread_line", "elo_diff", "off_diff", "def_diff"])

y = vegas_only["home_win"].to_numpy()  # same games in every row
pd.DataFrame({
    "Logistic + rest (ours)": _metrics(more["home_win"].to_numpy(), more["prob"]),
    "Vegas (raw)": _metrics(y, vegas_prob(vegas_only["spread_line"])),
    "Vegas spread alone": _metrics(y, vegas_only["prob"]),
    "Vegas + our features": _metrics(vegas_plus["home_win"].to_numpy(), vegas_plus["prob"]),
}).T

,games,accuracy,log_loss,brier
Logistic + rest (ours),2119.0,0.640868,0.634712,0.222168
Vegas (raw),2119.0,0.663992,0.610020,0.211111
Vegas spread alone,2119.0,0.663992,0.608303,0.210483
Vegas + our features,2119.0,0.665408,0.609539,0.211003


### **Takeaway**
- Adding features on top off the Vegas spread didn't improve predictions. 

## 5. Weekly Predictions
- Uses the trained models to predict upcoming games. 

In [28]:
feats3 = ["elo_diff", "off_diff", "def_diff"]

# Train on every finished game so far (all seasons, including 2026 to date)
train = data.dropna(subset=feats3 + ["home_score", "away_score"])
train = train[train["home_score"] != train["away_score"]].copy()  # drop ties
train["home_win"] = (train["home_score"] > train["away_score"]).astype(int)
final_model = LogisticRegression().fit(train[feats3], train["home_win"])

# Each team's current form = average of its last 6 finished games.
# Unlike the training data, there's no shift(1) here, because the game being
# predicted hasn't happened yet, so all 6 games are fair to use.
done = s.dropna(subset=["off_epa_pp", "def_epa_pp"])
form = (done.groupby("team").tail(6)
            .groupby("team")[["off_epa_pp", "def_epa_pp"]].mean())
form.head()

,off_epa_pp,def_epa_pp
team,,
ARI,-0.041467,0.134337
ATL,-0.057100,-0.015650
BAL,0.155279,0.023701
BUF,0.189340,0.109256
CAR,0.063919,0.041249


In [29]:
# Upcoming games = 2026 games with no score yet AND a date that hasn't passed.
# (Last night's game may still be missing its score in the data, so skip it.)
today = pd.Timestamp.today().normalize()
upcoming = df[df["home_score"].isna() & (df["season"] == 2026)].copy()
upcoming = upcoming[pd.to_datetime(upcoming["gameday"]) >= today]
next_week = upcoming["week"].min()
nxt = upcoming[upcoming["week"] == next_week].copy()

# Attach each team's current form, then build the same home-minus-away gaps as in training
nxt = nxt.join(form.add_prefix("home_"), on="home_team")
nxt = nxt.join(form.add_prefix("away_"), on="away_team")
nxt["elo_diff"] = nxt["elo_home"] - nxt["elo_away"]  # Elo ratings are already current
nxt["off_diff"] = nxt["home_off_epa_pp"] - nxt["away_off_epa_pp"]
nxt["def_diff"] = nxt["home_def_epa_pp"] - nxt["away_def_epa_pp"]

# Predicted probability that the home team wins
nxt["home_win_prob"] = final_model.predict_proba(nxt[feats3])[:, 1]

print("Predictions for week", next_week)
nxt[["gameday", "away_team", "home_team", "home_win_prob", "spread_line"]] \
    .sort_values("gameday").round(3)

Predictions for week 6


,gameday,away_team,home_team,home_win_prob,spread_line
3107,2026-10-15,SEA,DEN,0.449,-3.0
3108,2026-10-18,HOU,JAX,0.706,3.0
3109,2026-10-18,CHI,ATL,0.456,-3.0
3110,2026-10-18,BAL,CLE,0.334,-4.5
3111,2026-10-18,TEN,IND,0.760,7.0
3112,2026-10-18,NYJ,NE,0.785,7.5
3113,2026-10-18,NO,NYG,0.643,1.5
3114,2026-10-18,CAR,PHI,0.556,1.5
3115,2026-10-18,PIT,TB,0.480,-3.0
3116,2026-10-18,ARI,LA,0.796,13.5


In [30]:
import os
from datetime import datetime

os.makedirs("data/predictions", exist_ok=True)

In [31]:
# What Vegas implies for the home team, so model and market sit side by side
nxt["vegas_prob"] = vegas_prob(nxt["spread_line"])

out = nxt[["game_id", "season", "week", "gameday", "away_team", "home_team",
           "home_win_prob", "vegas_prob", "spread_line"]].copy()
out["predicted_at"] = datetime.now().strftime("%Y-%m-%d %H:%M")  # when the prediction was made

# One file per week, e.g. data/predictions/2026_week_06.csv
path = f"data/predictions/2026_week_{next_week:02d}.csv"
out.to_csv(path, index=False)
print("Saved", len(out), "predictions to", path)

Saved 14 predictions to data/predictions/2026_week_06.csv


## 6. Against the Spread (ATS)
- Building the Margin model and check how accurate the results are before using it for picks
- Vegas spread is hte margin prediction, so testing whether our model avg error reaches the spread

In [32]:
from sklearn.linear_model import Ridge

In [33]:
def walk_forward_margin(feats):
    """Walk-forward test, but predicting the home team's winning margin in points."""
    p = data.dropna(subset=feats + ["home_score", "away_score", "spread_line"]).copy()
    p["margin"] = p["home_score"] - p["away_score"]  # positive = home team won by that many

    out = []
    for season in range(2018, 2026):
        train = p[p["season"] < season]  # only the past
        test = p[(p["season"] == season) & (p["game_type"] == "REG")].copy()
        # Ridge is linear regression with a small penalty that keeps the weights stable
        model = Ridge(alpha=1.0).fit(train[feats], train["margin"])
        test["pred_margin"] = model.predict(test[feats])
        out.append(test)
    return pd.concat(out)

mres = walk_forward_margin(["elo_diff", "off_diff", "def_diff"])

In [34]:
# Vegas spread_line is positive when the home team is favored, so it is Vegas's predicted home margin
def errors(pred):
    e = mres["margin"] - pred
    return {"MAE (avg points off)": e.abs().mean(), "RMSE": (e ** 2).mean() ** 0.5}

pd.DataFrame({
    "Our margin model": errors(mres["pred_margin"]),
    "Vegas spread": errors(mres["spread_line"]),
}).T.round(2)

,MAE (avg points off),RMSE
Our margin model,10.22,13.19
Vegas spread,9.84,12.77


### Evaluation
- checking whether our 'disagreement' (margin vs spread) wins bets
- 'edge' is how much more (or less) we like the home team than the line does

In [35]:
# Edge = how many points our model likes the home team more than Vegas does
mres["edge"] = mres["pred_margin"] - mres["spread_line"]

# Drop pushes (margin exactly equals the spread): neither side wins
a = mres[mres["margin"] != mres["spread_line"]].copy()
a["home_covers"] = a["margin"] > a["spread_line"]  # home beats the spread
a["pick_home"] = a["edge"] > 0                     # we pick home if we like them more than Vegas
a["correct"] = a["pick_home"] == a["home_covers"]

print("All games:", round(a["correct"].mean(), 3), "on", len(a), "games")

# Does a bigger disagreement with Vegas mean a better hit rate?
a["edge_size"] = pd.cut(a["edge"].abs(), [0, 1, 2, 3, 100], labels=["0-1", "1-2", "2-3", "3+"])
a.groupby("edge_size", observed=True)["correct"].agg(["mean", "count"]).round(3)

All games: 0.481 on 2075 games


,mean,count
edge_size,,
0-1,0.429,559
1-2,0.498,490
2-3,0.488,389
3+,0.509,637


### **Takeaway**
- Model doesn't beat the spread, and larger edges  don't improve hit rate. 
- We're within noise of a coin flip. 

### Adding ATS layer to the Predictor

In [36]:
# Train the margin model on every finished game so far (same inputs as the win model)
tm = data.dropna(subset=feats3 + ["home_score", "away_score"]).copy()
tm["margin"] = tm["home_score"] - tm["away_score"]
margin_model = Ridge(alpha=1.0).fit(tm[feats3], tm["margin"])

# Predicted home margin and edge vs Vegas for next week's games
nxt["pred_margin"] = margin_model.predict(nxt[feats3])
nxt["edge"] = nxt["pred_margin"] - nxt["spread_line"]
nxt["ats_lean"] = nxt["home_team"].where(nxt["edge"] > 0, nxt["away_team"])  # the side we'd take

# Save with the win probabilities (overwrites this week's file with the extra columns)
out = nxt[["game_id", "season", "week", "gameday", "away_team", "home_team",
           "home_win_prob", "vegas_prob", "spread_line",
           "pred_margin", "edge", "ats_lean"]].copy()
out["predicted_at"] = datetime.now().strftime("%Y-%m-%d %H:%M")
out.to_csv(path, index=False)

out.sort_values("edge", key=abs, ascending=False).round(2)

,game_id,season,week,gameday,away_team,home_team,home_win_prob,vegas_prob,spread_line,pred_margin,edge,ats_lean,predicted_at
3108,2026_06_HOU_JAX,2026,6,2026-10-18,HOU,JAX,0.71,0.59,3.0,6.57,3.57,JAX,2026-10-13 02:39
3120,2026_06_WAS_SF,2026,6,2026-10-19,WAS,SF,0.81,0.71,7.5,10.75,3.25,SF,2026-10-13 02:39
3113,2026_06_NO_NYG,2026,6,2026-10-18,NO,NYG,0.64,0.54,1.5,4.71,3.21,NYG,2026-10-13 02:39
3116,2026_06_ARI_LA,2026,6,2026-10-18,ARI,LA,0.80,0.84,13.5,10.43,-3.07,ARI,2026-10-13 02:39
3115,2026_06_PIT_TB,2026,6,2026-10-18,PIT,TB,0.48,0.41,-3.0,-0.02,2.98,TB,2026-10-13 02:39
3119,2026_06_DAL_GB,2026,6,2026-10-18,DAL,GB,0.62,0.54,1.5,4.08,2.58,GB,2026-10-13 02:39
3109,2026_06_CHI_ATL,2026,6,2026-10-18,CHI,ATL,0.46,0.41,-3.0,-0.91,2.09,ATL,2026-10-13 02:39
3112,2026_06_NYJ_NE,2026,6,2026-10-18,NYJ,NE,0.79,0.71,7.5,9.46,1.96,NE,2026-10-13 02:39
3111,2026_06_TEN_IND,2026,6,2026-10-18,TEN,IND,0.76,0.70,7.0,8.67,1.67,IND,2026-10-13 02:39
3107,2026_06_SEA_DEN,2026,6,2026-10-15,SEA,DEN,0.45,0.41,-3.0,-1.45,1.55,DEN,2026-10-13 02:39


## 7. Calibration Check
- Checks whether the confidence number from the probabilities means anything to be trusted.
- Don't want to just stop at Accuracy

In [38]:
import numpy as np

In [39]:
# Use the backtest predictions: one row per game, with our home win probability in "prob"
cal = base.copy()

# Our pick and confidence, same logic as the dashboard
cal["pick_home"] = cal["prob"] > 0.5
cal["confidence"] = np.where(cal["pick_home"], cal["prob"], 1 - cal["prob"])

# Did the pick win? (home_win is 1 if the home team won)
cal["correct"] = cal["pick_home"] == (cal["home_win"] == 1)

# Group games into confidence bands, then compare what the model said with what happened
cal["band"] = pd.cut(cal["confidence"], [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 1.0],
                     labels=["50-55%", "55-60%", "60-65%", "65-70%", "70-75%", "75%+"])

table = cal.groupby("band", observed=True).agg(
    games=("correct", "count"),
    avg_confidence=("confidence", "mean"),  # what the model claimed
    actual_win_rate=("correct", "mean"),    # what actually happened
).round(3)

table

,games,avg_confidence,actual_win_rate
band,,,
50-55%,411,0.524,0.523
55-60%,398,0.575,0.573
60-65%,376,0.625,0.614
65-70%,290,0.672,0.686
70-75%,256,0.725,0.746
75%+,388,0.810,0.763


### **Takeaway**
- Model is well calibrated so we don't need to do anything to it.
- Most bands, the confidence is within a couple points of the actual win rate. 